In [3]:
# =============================================================================
# ПОЛНЫЙ АВТОНОМНЫЙ КОД: ПОДВОДНАЯ СИСТЕМА ОБНАРУЖЕНИЯ УТОПАЮЩИХ
# =============================================================================

import os
import json
import yaml
import time
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from collections import defaultdict, deque
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, FileLink
from ultralytics import YOLO
import warnings
import threading
warnings.filterwarnings('ignore')

print("📚 Библиотеки импортированы")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🎮 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

# =============================================================================
# КЛАСС ProjectState
# =============================================================================

class ProjectState:
    def __init__(self):
        self.BASE_DIR = Path.cwd()
        self.DATASET_DIR = self.BASE_DIR / "dataset"
        self.MODELS_DIR = self.BASE_DIR / "models"
        self.VIDEOS_DIR = self.BASE_DIR / "videos"
        self.OUTPUTS_DIR = self.BASE_DIR / "outputs"
        self.CONFIGS_DIR = self.BASE_DIR / "configs"
        self.STATE_FILE = self.CONFIGS_DIR / "project_state.json"
        
        for dir_path in [self.DATASET_DIR, self.MODELS_DIR, self.VIDEOS_DIR, 
                         self.OUTPUTS_DIR, self.CONFIGS_DIR]:
            dir_path.mkdir(exist_ok=True, parents=True)
        
        for split in ['train', 'val']:
            (self.DATASET_DIR / "images" / split).mkdir(exist_ok=True, parents=True)
            (self.DATASET_DIR / "labels" / split).mkdir(exist_ok=True, parents=True)
        
        self.state = self._load_state()
    
    def _load_state(self):
        default = {
            'dataset_checked': False,
            'config_created': False,
            'model_trained': False,
            'best_model_path': None,
            'last_training': None,
            'class_names': ['swimming', 'struggling', 'drowning']
        }
        if self.STATE_FILE.exists():
            try:
                with open(self.STATE_FILE, 'r') as f:
                    default.update(json.load(f))
            except:
                pass
        return default
    
    def save_state(self):
        with open(self.STATE_FILE, 'w') as f:
            json.dump(self.state, f, indent=2)
    
    def update_state(self, **kwargs):
        self.state.update(kwargs)
        self.save_state()

# =============================================================================
# КЛАСС DatasetManager
# =============================================================================

class DatasetManager:
    def __init__(self, project_state):
        self.project = project_state
        self.class_names = ['swimming', 'struggling', 'drowning']
    
    def check_dataset_structure(self):
        print("🔍 Проверка структуры датасета...")
        total_images = 0
        total_labels = 0
        class_counts = [0, 0, 0]
        
        for split in ['train', 'val']:
            img_dir = self.project.DATASET_DIR / "images" / split
            lbl_dir = self.project.DATASET_DIR / "labels" / split
            images = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
            labels = list(lbl_dir.glob("*.txt"))
            
            print(f"  {split}: {len(images)} изображений, {len(labels)} аннотаций")
            total_images += len(images)
            total_labels += len(labels)
            
            for lbl_file in labels:
                try:
                    with open(lbl_file, 'r') as f:
                        for line in f:
                            class_id = int(line.strip().split()[0])
                            if class_id < 3:
                                class_counts[class_id] += 1
                except:
                    pass
        
        print(f"\n📊 Всего: {total_images} изображений, {total_labels} аннотаций")
        print(f"📊 Распределение классов:")
        for i, name in enumerate(self.class_names):
            print(f"   {name}: {class_counts[i]}")
        
        self.project.update_state(dataset_checked=True)
        return {'total_images': total_images, 'total_labels': total_labels, 'class_counts': class_counts}
    
    def create_data_yaml(self):
        data_config = {
            'path': str(self.project.DATASET_DIR.absolute()),
            'train': 'images/train',
            'val': 'images/val',
            'nc': 3,
            'names': self.class_names
        }
        yaml_path = self.project.CONFIGS_DIR / "dataset.yaml"
        with open(yaml_path, 'w') as f:
            yaml.dump(data_config, f)
        print(f"✅ data.yaml создан: {yaml_path}")
        self.project.update_state(config_created=True)
        return yaml_path
    
    def show_samples(self, split='train', num_samples=5):
        img_dir = self.project.DATASET_DIR / "images" / split
        images = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
        if not images:
            print("❌ Нет изображений")
            return
        import random
        samples = random.sample(images, min(num_samples, len(images)))
        fig, axes = plt.subplots(1, len(samples), figsize=(15,5))
        if len(samples) == 1:
            axes = [axes]
        for i, img_path in enumerate(samples):
            img = cv2.imread(str(img_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[i].imshow(img)
            axes[i].axis('off')
            axes[i].set_title(img_path.name)
        plt.show()

# =============================================================================
# КЛАСС ModelTrainer
# =============================================================================

class ModelTrainer:
    def __init__(self, project_state):
        self.project = project_state
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.training_active = False
    
    def train_model(self, model_size='s', epochs=100, batch_size=16, imgsz=640):
        print("="*60)
        print("🚀 ОБУЧЕНИЕ МОДЕЛИ")
        print("="*60)
        
        yaml_path = self.project.CONFIGS_DIR / "dataset.yaml"
        if not yaml_path.exists():
            print("❌ Сначала создайте data.yaml во вкладке 'Датасет'!")
            return None
        
        print(f"📊 Устройство: {self.device}")
        print(f"📊 Модель: YOLOv8{model_size}")
        print(f"📊 Эпохи: {epochs}")
        print(f"📊 Batch size: {batch_size}")
        print(f"📊 Размер: {imgsz}")
        
        try:
            model = YOLO(f'yolov8{model_size}.pt')
            
            results = model.train(
                data=str(yaml_path),
                epochs=epochs,
                imgsz=imgsz,
                batch=batch_size,
                device=self.device,
                patience=20,
                save=True,
                project=str(self.project.MODELS_DIR),
                name='drowning_detection',
                exist_ok=True,
                pretrained=True,
                optimizer='AdamW',
                lr0=0.001,
                seed=42,
                verbose=True
            )
            
            best_path = self.project.MODELS_DIR / 'drowning_detection' / 'weights' / 'best.pt'
            self.project.update_state(
                model_trained=True,
                best_model_path=str(best_path),
                last_training=datetime.now().isoformat()
            )
            
            print(f"\n✅ Модель сохранена: {best_path}")
            return best_path
            
        except Exception as e:
            print(f"\n❌ Ошибка: {e}")
            return None
    
    def evaluate_model(self):
        path = self.project.state.get('best_model_path')
        yaml = self.project.CONFIGS_DIR / "dataset.yaml"
        
        if not path or not Path(path).exists():
            print("❌ Модель не найдена!")
            return
        
        print("\n📊 ОЦЕНКА МОДЕЛИ")
        print("="*50)
        
        model = YOLO(path)
        metrics = model.val(data=str(yaml), verbose=False)
        
        if hasattr(metrics, 'box'):
            print(f"🎯 mAP50: {metrics.box.map50:.4f}")
            print(f"🎯 mAP50-95: {metrics.box.map:.4f}")
            print(f"📊 Precision: {metrics.box.p.mean():.4f}")
            print(f"📊 Recall: {metrics.box.r.mean():.4f}")

# =============================================================================
# УЛУЧШЕННЫЙ КЛАСС PoseAnalyzer с анализом пассивного движения
# =============================================================================

class PoseAnalyzer:
    def __init__(self):
        self.NOSE = 0
        self.LEFT_EYE = 1
        self.RIGHT_EYE = 2
        self.LEFT_EAR = 3
        self.RIGHT_EAR = 4
        self.LEFT_SHOULDER = 5
        self.RIGHT_SHOULDER = 6
        self.LEFT_ELBOW = 7
        self.RIGHT_ELBOW = 8
        self.LEFT_WRIST = 9
        self.RIGHT_WRIST = 10
        self.LEFT_HIP = 11
        self.RIGHT_HIP = 12
        self.LEFT_KNEE = 13
        self.RIGHT_KNEE = 14
        self.LEFT_ANKLE = 15
        self.RIGHT_ANKLE = 16
        
        self.SWIMMING_ANGLE_THRESHOLD = 30
        self.DROWNING_ANGLE_THRESHOLD = 60
        self.MIN_KEYPOINTS = 10
    
    def is_human(self, keypoints, confidence_threshold=0.3):
        """Проверка, является ли объект человеком"""
        if keypoints is None or len(keypoints) < 51:
            return False
        
        visible = 0
        for i in range(17):
            if i*3+2 < len(keypoints):
                conf = keypoints[i*3 + 2]
                if conf > confidence_threshold:
                    visible += 1
        
        return visible >= self.MIN_KEYPOINTS
    
    def calculate_body_angle(self, keypoints):
        """Вычисление угла наклона тела"""
        if keypoints is None or len(keypoints) < 51:
            return None
        
        try:
            left_shoulder_x = keypoints[self.LEFT_SHOULDER*3]
            left_shoulder_y = keypoints[self.LEFT_SHOULDER*3 + 1]
            right_shoulder_x = keypoints[self.RIGHT_SHOULDER*3]
            right_shoulder_y = keypoints[self.RIGHT_SHOULDER*3 + 1]
            
            left_hip_x = keypoints[self.LEFT_HIP*3]
            left_hip_y = keypoints[self.LEFT_HIP*3 + 1]
            right_hip_x = keypoints[self.RIGHT_HIP*3]
            right_hip_y = keypoints[self.RIGHT_HIP*3 + 1]
            
            # Проверка уверенности ключевых точек
            if (keypoints[self.LEFT_SHOULDER*3+2] < 0.3 or 
                keypoints[self.RIGHT_SHOULDER*3+2] < 0.3 or
                keypoints[self.LEFT_HIP*3+2] < 0.3 or 
                keypoints[self.RIGHT_HIP*3+2] < 0.3):
                return None
            
            # Центры плеч и бедер
            shoulder_center_x = (left_shoulder_x + right_shoulder_x) / 2
            shoulder_center_y = (left_shoulder_y + right_shoulder_y) / 2
            hip_center_x = (left_hip_x + right_hip_x) / 2
            hip_center_y = (left_hip_y + right_hip_y) / 2
            
            dy = shoulder_center_y - hip_center_y
            dx = shoulder_center_x - hip_center_x
            
            if abs(dx) < 1e-6:
                return 90
            
            angle = abs(np.arctan2(dy, dx) * 180 / np.pi)
            return min(90, angle)
            
        except Exception as e:
            return None
    
    def analyze_arm_position(self, keypoints):
        """
        Анализ положения рук
        Возвращает: (arms_up, confidence, arm_orientation)
        arm_orientation: 'vertical', 'horizontal', 'mixed'
        """
        if keypoints is None or len(keypoints) < 51:
            return False, 0.0, 'unknown'
        
        try:
            # Проверяем левую руку
            left_shoulder = keypoints[self.LEFT_SHOULDER*3:self.LEFT_SHOULDER*3+3]
            left_elbow = keypoints[self.LEFT_ELBOW*3:self.LEFT_ELBOW*3+3]
            left_wrist = keypoints[self.LEFT_WRIST*3:self.LEFT_WRIST*3+3]
            
            # Проверяем правую руку
            right_shoulder = keypoints[self.RIGHT_SHOULDER*3:self.RIGHT_SHOULDER*3+3]
            right_elbow = keypoints[self.RIGHT_ELBOW*3:self.RIGHT_ELBOW*3+3]
            right_wrist = keypoints[self.RIGHT_WRIST*3:self.RIGHT_WRIST*3+3]
            
            # Пороги уверенности для точек руки
            arm_conf_threshold = 0.3
            
            left_arm_visible = (left_shoulder[2] > arm_conf_threshold and 
                              left_elbow[2] > arm_conf_threshold and 
                              left_wrist[2] > arm_conf_threshold)
            
            right_arm_visible = (right_shoulder[2] > arm_conf_threshold and 
                               right_elbow[2] > arm_conf_threshold and 
                               right_wrist[2] > arm_conf_threshold)
            
            if not (left_arm_visible or right_arm_visible):
                return False, 0.0, 'unknown'
            
            arms_up_score = 0
            arms_checked = 0
            arm_orientations = []
            
            # Анализ левой руки
            if left_arm_visible:
                # Вектор от плеча к локтю
                shoulder_to_elbow = np.array([left_elbow[0] - left_shoulder[0], 
                                             left_elbow[1] - left_shoulder[1]])
                
                # Вычисляем угол подъема руки относительно горизонтали
                if abs(shoulder_to_elbow[0]) > 1e-6:
                    arm_angle = abs(np.arctan2(shoulder_to_elbow[1], shoulder_to_elbow[0]) * 180 / np.pi)
                else:
                    arm_angle = 90
                
                # Определяем ориентацию
                if arm_angle > 60:
                    arm_orientations.append('vertical')
                    if shoulder_to_elbow[1] < -10:  # Рука поднята вверх
                        arms_up_score += 1
                elif arm_angle < 30:
                    arm_orientations.append('horizontal')
                else:
                    arm_orientations.append('diagonal')
                
                arms_checked += 1
            
            # Анализ правой руки
            if right_arm_visible:
                shoulder_to_elbow = np.array([right_elbow[0] - right_shoulder[0], 
                                             right_elbow[1] - right_shoulder[1]])
                
                if abs(shoulder_to_elbow[0]) > 1e-6:
                    arm_angle = abs(np.arctan2(shoulder_to_elbow[1], shoulder_to_elbow[0]) * 180 / np.pi)
                else:
                    arm_angle = 90
                
                if arm_angle > 60:
                    arm_orientations.append('vertical')
                    if shoulder_to_elbow[1] < -10:
                        arms_up_score += 1
                elif arm_angle < 30:
                    arm_orientations.append('horizontal')
                else:
                    arm_orientations.append('diagonal')
                
                arms_checked += 1
            
            # Определяем общую ориентацию рук
            if arms_checked > 0:
                if arm_orientations.count('vertical') / arms_checked > 0.7:
                    overall_orientation = 'vertical'
                elif arm_orientations.count('horizontal') / arms_checked > 0.7:
                    overall_orientation = 'horizontal'
                else:
                    overall_orientation = 'mixed'
            else:
                overall_orientation = 'unknown'
            
            confidence = arms_up_score / arms_checked if arms_checked > 0 else 0
            return confidence > 0.5, confidence, overall_orientation
            
        except Exception as e:
            return False, 0.0, 'unknown'
    
    def analyze_pose(self, keypoints):
        """
        Анализ позы человека
        Возвращает: (class_id, confidence, pose_features)
        pose_features: словарь с дополнительными признаками
        """
        if not self.is_human(keypoints):
            return None, 0.0, {}
        
        angle = self.calculate_body_angle(keypoints)
        arms_up, arms_confidence, arm_orientation = self.analyze_arm_position(keypoints)
        
        # Проверяем видимость головы
        head_indices = [0, 1, 2, 3, 4]
        visible_head = 0
        
        for idx in head_indices:
            if idx*3+2 < len(keypoints):
                conf = keypoints[idx*3 + 2]
                if conf > 0.3:
                    visible_head += 1
        
        head_visible = visible_head >= 3
        
        # Собираем все признаки
        pose_features = {
            'head_visible': head_visible,
            'body_angle': angle,
            'arms_up': arms_up,
            'arms_confidence': arms_confidence,
            'arm_orientation': arm_orientation,
            'keypoints_count': sum(1 for i in range(17) if i*3+2 < len(keypoints) and keypoints[i*3+2] > 0.3)
        }
        
        return 0, 0.5, pose_features  # Временное значение, реальный класс определит BehaviorAnalyzer


# =============================================================================
# ИСПРАВЛЕННЫЙ КЛАСС UnderwaterBehaviorAnalyzer с защитой 0.75 сек
# =============================================================================

class UnderwaterBehaviorAnalyzer:
    def __init__(self, fps=30):
        self.fps = fps
        self.DROWNING_CONFIRMATION_TIME = 0.5  # 0.5 секунды для определения класса
        self.DROWNING_ALERT_TIME = 0.75  # 0.75 секунды для алерта
        self.MIN_DROWNING_FRAMES = int(fps * 0.5)
        
    def is_head_visible(self, keypoints, confidence_threshold=0.3):
        """Проверка, видна ли голова человека"""
        if keypoints is None or len(keypoints) < 51:
            return False
        
        head_indices = [0, 1, 2, 3, 4]
        visible_head_points = 0
        
        for idx in head_indices:
            if idx*3+2 < len(keypoints):
                conf = keypoints[idx*3 + 2]
                if conf > confidence_threshold:
                    visible_head_points += 1
        
        return visible_head_points >= 3
    
    def analyze_vertical_movement(self, history):
        """
        Анализ вертикального движения (падения)
        Возвращает: (is_falling, falling_speed, is_passive)
        """
        if len(history) < self.fps // 2:
            return False, 0, False
        
        recent = history[-int(self.fps * 0.5):]  # Последние 0.5 секунды
        
        if len(recent) < 2:
            return False, 0, False
        
        # Анализируем движение по Y координате
        y_positions = []
        for h in recent:
            center_y = (h['bbox'][1] + h['bbox'][3]) / 2
            y_positions.append(center_y)
        
        y_changes = np.diff(y_positions)
        avg_y_speed = np.mean(np.abs(y_changes)) if len(y_changes) > 0 else 0
        
        consistent_direction = all(y_changes > 0) if len(y_changes) > 0 else False
        
        # Проверяем горизонтальные движения
        x_positions = []
        for h in recent:
            center_x = (h['bbox'][0] + h['bbox'][2]) / 2
            x_positions.append(center_x)
        
        x_changes = np.diff(x_positions)
        avg_x_speed = np.mean(np.abs(x_changes)) if len(x_changes) > 0 else 0
        
        is_falling = avg_y_speed > 5 and consistent_direction
        is_passive = is_falling and avg_x_speed < 3
        
        return is_falling, avg_y_speed, is_passive
    
    def analyze_movement_decay(self, history):
        """Анализ затухания движений во времени"""
        if len(history) < self.fps:
            return False, 0.0
        
        recent_history = history[-int(self.fps * 2):]
        
        if len(recent_history) < self.fps // 2:
            return False, 0.0
        
        mid = len(recent_history) // 2
        first_half = recent_history[:mid]
        second_half = recent_history[mid:]
        
        first_speeds = []
        for i in range(1, len(first_half)):
            prev = first_half[i-1]['bbox']
            curr = first_half[i]['bbox']
            prev_center = [(prev[0] + prev[2])/2, (prev[1] + prev[3])/2]
            curr_center = [(curr[0] + curr[2])/2, (curr[1] + curr[3])/2]
            speed = np.sqrt((curr_center[0] - prev_center[0])**2 + (curr_center[1] - prev_center[1])**2)
            first_speeds.append(speed)
        
        second_speeds = []
        for i in range(1, len(second_half)):
            prev = second_half[i-1]['bbox']
            curr = second_half[i]['bbox']
            prev_center = [(prev[0] + prev[2])/2, (prev[1] + prev[3])/2]
            curr_center = [(curr[0] + curr[2])/2, (curr[1] + curr[3])/2]
            speed = np.sqrt((curr_center[0] - prev_center[0])**2 + (curr_center[1] - prev_center[1])**2)
            second_speeds.append(speed)
        
        avg_first = np.mean(first_speeds) if first_speeds else 0
        avg_second = np.mean(second_speeds) if second_speeds else 0
        
        if avg_first > 0:
            total_decay = (avg_first - avg_second) / avg_first
        else:
            total_decay = 0
        
        is_decaying = total_decay > 0.5 and avg_second < 5
        
        return is_decaying, total_decay
    
    def calculate_movement(self, history, current_bbox):
        """Анализ движения на основе истории позиций"""
        if len(history) < self.fps // 2:
            return True, 1.0, 0, 0
        
        recent = history[-self.fps//2:]
        if len(recent) < 2:
            return True, 1.0, 0, 0
        
        first_center = [
            (recent[0]['bbox'][0] + recent[0]['bbox'][2]) / 2,
            (recent[0]['bbox'][1] + recent[0]['bbox'][3]) / 2
        ]
        
        last_center = [
            (current_bbox[0] + current_bbox[2]) / 2,
            (current_bbox[1] + current_bbox[3]) / 2
        ]
        
        displacement = np.sqrt(
            (last_center[0] - first_center[0])**2 + 
            (last_center[1] - first_center[1])**2
        )
        
        vertical_displacement = abs(last_center[1] - first_center[1])
        horizontal_displacement = abs(last_center[0] - first_center[0])
        
        first_height = recent[0]['bbox'][3] - recent[0]['bbox'][1]
        current_height = current_bbox[3] - current_bbox[1]
        height_change = abs(current_height - first_height) / first_height if first_height > 0 else 0
        
        is_moving = displacement > 10 or height_change > 0.2
        speed = displacement / len(recent)
        
        return is_moving, speed, vertical_displacement, horizontal_displacement
    
    def analyze_underwater_behavior(self, track, current_time, angle=None):
        """
        АНАЛИЗ ПОВЕДЕНИЯ С ОПРЕДЕЛЕНИЕМ КЛАССА
        """
        
        if len(track['history']) < self.fps // 2:
            return track.get('class', 0), 0.5
        
        current_bbox = track['bbox']
        history = track['history']
        
        last_keypoints = track.get('last_keypoints')
        head_visible = self.is_head_visible(last_keypoints)
        
        is_moving, speed, vert_move, horiz_move = self.calculate_movement(history, current_bbox)
        is_falling, fall_speed, is_passive_fall = self.analyze_vertical_movement(history)
        is_decaying, decay_rate = self.analyze_movement_decay(history)
        
        pose_analyzer = PoseAnalyzer()
        arms_up, arms_confidence, arm_orientation = pose_analyzer.analyze_arm_position(last_keypoints) if last_keypoints is not None else (False, 0.0, 'unknown')
        
        # Вычисляем время без активных движений
        time_without_movement = 0
        recent_history = history[-int(self.fps * 2):]
        
        for i, h in enumerate(reversed(recent_history)):
            if i == 0:
                continue
            prev_h = recent_history[-(i+1)]
            
            prev_center = [(prev_h['bbox'][0] + prev_h['bbox'][2])/2, 
                          (prev_h['bbox'][1] + prev_h['bbox'][3])/2]
            curr_center = [(h['bbox'][0] + h['bbox'][2])/2, 
                          (h['bbox'][1] + h['bbox'][3])/2]
            
            movement = np.sqrt((curr_center[0] - prev_center[0])**2 + 
                              (curr_center[1] - prev_center[1])**2)
            
            if movement < 5:
                time_without_movement += 1 / self.fps
            else:
                break
        
        # Собираем все признаки
        features = {
            'head_visible': head_visible,
            'is_moving': is_moving,
            'is_falling': is_falling,
            'is_passive_fall': is_passive_fall,
            'is_decaying': is_decaying,
            'arms_up': arms_up,
            'arm_orientation': arm_orientation,
            'time_without_movement': time_without_movement,
            'vert_move': vert_move,
            'horiz_move': horiz_move
        }
        
        if 'feature_history' not in track:
            track['feature_history'] = []
        
        track['feature_history'].append(features)
        if len(track['feature_history']) > self.fps * 3:
            track['feature_history'].pop(0)
        
        # Анализируем последние 0.5 секунды для определения класса
        confirmation_frames = max(1, int(self.fps * self.DROWNING_CONFIRMATION_TIME))
        
        if len(track['feature_history']) < confirmation_frames:
            return track.get('class', 0), 0.5
        
        recent_features = track['feature_history'][-confirmation_frames:]
        
        # Определяем класс
        drowning_score = 0
        struggling_score = 0
        swimming_score = 0
        
        for f in recent_features:
            if f.get('is_passive_fall', False) and f.get('arm_orientation') == 'vertical':
                drowning_score += 3
            elif f.get('is_falling', False) and not f.get('is_moving', True) and f.get('vert_move', 0) > 15:
                drowning_score += 2
            elif f.get('time_without_movement', 0) > 1.0 and f.get('arms_up', False):
                drowning_score += 2
            elif f.get('is_decaying', False) and not f.get('is_moving', True):
                drowning_score += 2
            
            elif f.get('is_falling', False) and f.get('is_moving', False):
                struggling_score += 2
            elif f.get('arms_up', False) and f.get('is_moving', False):
                struggling_score += 2
            elif f.get('is_moving', False) and f.get('vert_move', 0) > f.get('horiz_move', 0):
                struggling_score += 1
            
            elif f.get('is_moving', False) and f.get('horiz_move', 0) > f.get('vert_move', 0):
                swimming_score += 2
            elif f.get('head_visible', False) and f.get('is_moving', False):
                swimming_score += 1
            else:
                swimming_score += 1
        
        scores = {0: swimming_score, 1: struggling_score, 2: drowning_score}
        max_class = max(scores, key=scores.get)
        total = sum(scores.values())
        
        if total > 0:
            confidence = scores[max_class] / total
            return max_class, confidence
        
        return track.get('class', 0), 0.5


# =============================================================================
# ИСПРАВЛЕННЫЙ КЛАСС SmartUnderwaterTracker с залипанием состояния drowning
# =============================================================================

class SmartUnderwaterTracker:
    def __init__(self, fps=30):
        self.tracks = {}
        self.next_id = 0
        self.max_lost = 30
        self.fps = fps
        self.pose_analyzer = PoseAnalyzer()
        self.behavior_analyzer = UnderwaterBehaviorAnalyzer(fps)
        
        self.colors = {
            0: (0, 255, 0),      # swimming - зеленый
            1: (0, 255, 255),     # struggling - желтый
            2: (0, 0, 255),       # drowning - красный
        }
        
        self.box_thickness = 2
        self.iou_threshold = 0.3
        
        # Для отслеживания состояния drowning
        self.drowning_start_time = {}  # Время начала drowning
        self.drowning_confirmed = {}   # Флаг подтвержденного drowning (залипание)
        self.last_class = {}           # Последний класс перед drowning
    
    def _merge_duplicate_detections(self, detections, keypoints_list):
        """Объединение дублирующихся детекций"""
        if len(detections) <= 1:
            return detections, keypoints_list
        
        merged_indices = set()
        unique_detections = []
        unique_keypoints = []
        
        for i in range(len(detections)):
            if i in merged_indices:
                continue
            
            x1, y1, x2, y2, conf_i, _ = detections[i]
            
            overlapping = [i]
            for j in range(i+1, len(detections)):
                if j in merged_indices:
                    continue
                
                ox1, oy1, ox2, oy2, conf_j, _ = detections[j]
                
                ix1 = max(x1, ox1)
                iy1 = max(y1, oy1)
                ix2 = min(x2, ox2)
                iy2 = min(y2, oy2)
                
                if ix2 > ix1 and iy2 > iy1:
                    inter = (ix2 - ix1) * (iy2 - iy1)
                    area_i = (x2 - x1) * (y2 - y1)
                    area_j = (ox2 - ox1) * (oy2 - oy1)
                    iou = inter / (area_i + area_j - inter + 1e-6)
                    
                    if iou > 0.3:
                        overlapping.append(j)
                        merged_indices.add(j)
            
            if len(overlapping) > 1:
                total_conf = 0
                sum_x1, sum_y1, sum_x2, sum_y2 = 0, 0, 0, 0
                best_kp = None
                max_conf = 0
                
                for idx in overlapping:
                    conf = detections[idx][4]
                    total_conf += conf
                    sum_x1 += detections[idx][0] * conf
                    sum_y1 += detections[idx][1] * conf
                    sum_x2 += detections[idx][2] * conf
                    sum_y2 += detections[idx][3] * conf
                    
                    if idx < len(keypoints_list) and conf > max_conf:
                        max_conf = conf
                        best_kp = keypoints_list[idx]
                
                avg_x1 = sum_x1 / total_conf
                avg_y1 = sum_y1 / total_conf
                avg_x2 = sum_x2 / total_conf
                avg_y2 = sum_y2 / total_conf
                
                unique_detections.append([avg_x1, avg_y1, avg_x2, avg_y2, max_conf, 0])
                unique_keypoints.append(best_kp if best_kp is not None else keypoints_list[overlapping[0]])
            else:
                unique_detections.append(detections[i])
                if i < len(keypoints_list):
                    unique_keypoints.append(keypoints_list[i])
            
            merged_indices.add(i)
        
        return unique_detections, unique_keypoints
    
    def update(self, frame, detections, keypoints_list, current_time):
        """Обновление трекера с новыми детекциями и залипанием состояния drowning"""
        used = set()
        
        if detections and keypoints_list:
            detections, keypoints_list = self._merge_duplicate_detections(detections, keypoints_list)
        
        for det_idx, det in enumerate(detections):
            x1, y1, x2, y2, conf, _ = det
            keypoints = keypoints_list[det_idx] if keypoints_list and det_idx < len(keypoints_list) else None
            
            if keypoints is None or not self.pose_analyzer.is_human(keypoints):
                continue
            
            angle = self.pose_analyzer.calculate_body_angle(keypoints)
            
            best_id = None
            best_iou = 0.3
            best_iou_value = 0
            
            for tid, track in self.tracks.items():
                if tid in used:
                    continue
                
                tx1, ty1, tx2, ty2 = track['bbox']
                
                ix1 = max(x1, tx1)
                iy1 = max(y1, ty1)
                ix2 = min(x2, tx2)
                iy2 = min(y2, ty2)
                
                if ix2 > ix1 and iy2 > iy1:
                    inter = (ix2 - ix1) * (iy2 - iy1)
                    area1 = (x2 - x1) * (y2 - y1)
                    area2 = (tx2 - tx1) * (ty2 - ty1)
                    iou = inter / (area1 + area2 - inter + 1e-6)
                    
                    if iou > best_iou and iou > best_iou_value:
                        best_iou_value = iou
                        best_id = tid
            
            if best_id is not None:
                track = self.tracks[best_id]
                track['bbox'] = [x1, y1, x2, y2]
                track['last_keypoints'] = keypoints
                track['lost'] = 0
                track['last_seen'] = current_time
                
                # Проверяем было ли движение
                was_moving = False
                if len(track['history']) > 0:
                    prev = track['history'][-1]['bbox']
                    prev_center = [(prev[0] + prev[2])/2, (prev[1] + prev[3])/2]
                    curr_center = [(x1 + x2)/2, (y1 + y2)/2]
                    movement = np.sqrt((curr_center[0] - prev_center[0])**2 + (curr_center[1] - prev_center[1])**2)
                    was_moving = movement > 10
                
                track['history'].append({
                    'time': current_time,
                    'bbox': [x1, y1, x2, y2],
                    'keypoints': keypoints,
                    'angle': angle,
                    'class': track.get('class', 0),
                    'was_moving': was_moving
                })
                
                if len(track['history']) > self.fps * 10:
                    track['history'].pop(0)
                
                # Получаем новый класс от анализатора
                new_class, confidence = self.behavior_analyzer.analyze_underwater_behavior(
                    track, current_time, angle
                )
                
                # ========== ЛОГИКА ЗАЛИПАНИЯ DROWNING ==========
                tid = best_id
                
                # Инициализируем состояние для нового трека
                if tid not in self.drowning_confirmed:
                    self.drowning_confirmed[tid] = False
                if tid not in self.last_class:
                    self.last_class[tid] = 0
                
                # Если drowning уже подтвержден - держим класс 2
                if self.drowning_confirmed[tid]:
                    track['class'] = 2
                    track['conf'] = 0.95
                    print(f"⚠️ Трек {tid}: DROWNING ЗАЛИП (подтвержден ранее)")
                else:
                    # Проверяем, нужно ли подтвердить drowning
                    if new_class == 2:
                        # Начало или продолжение drowning
                        if tid not in self.drowning_start_time:
                            self.drowning_start_time[tid] = current_time
                            print(f"🔴 Трек {tid}: Начало drowning в {current_time:.2f}")
                        
                        # Вычисляем длительность
                        drowning_duration = current_time - self.drowning_start_time[tid]
                        
                        # Если длится больше 0.75 секунды - подтверждаем
                        if drowning_duration >= 0.75:
                            self.drowning_confirmed[tid] = True
                            track['class'] = 2
                            track['conf'] = 0.95
                            print(f"🚨 Трек {tid}: DROWNING ПОДТВЕРЖДЕН через {drowning_duration:.2f}с")
                        else:
                            # Еще не подтвержден, используем текущий класс
                            track['class'] = new_class
                            track['conf'] = confidence
                    else:
                        # Не drowning - сбрасываем таймер
                        if tid in self.drowning_start_time:
                            del self.drowning_start_time[tid]
                        
                        # Используем текущий класс
                        track['class'] = new_class
                        track['conf'] = confidence
                
                self.last_class[tid] = track['class']
                used.add(tid)
            else:
                # Новый трек
                self.tracks[self.next_id] = {
                    'bbox': [x1, y1, x2, y2],
                    'class': 0,
                    'conf': 0.5,
                    'lost': 0,
                    'last_seen': current_time,
                    'last_keypoints': keypoints,
                    'history': [{
                        'time': current_time,
                        'bbox': [x1, y1, x2, y2],
                        'keypoints': keypoints,
                        'angle': angle,
                        'class': 0,
                        'was_moving': False
                    }],
                    'alert': False,
                    'feature_history': [],
                    'last_alert_time': 0
                }
                self.next_id += 1
        
        # Очистка потерянных треков
        for tid in list(self.tracks.keys()):
            if tid not in used:
                self.tracks[tid]['lost'] += 1
                if self.tracks[tid]['lost'] > self.max_lost:
                    # Трек потерян - очищаем его состояние
                    del self.tracks[tid]
                    if tid in self.drowning_start_time:
                        del self.drowning_start_time[tid]
                    if tid in self.drowning_confirmed:
                        del self.drowning_confirmed[tid]
                    if tid in self.last_class:
                        del self.last_class[tid]
    
    def draw(self, frame):
        """Отрисовка результатов на кадре с индикацией подтвержденного drowning"""
        current_time = time.time()
        
        for tid, track in self.tracks.items():
            if track['lost'] > 0:
                continue
            
            x1, y1, x2, y2 = map(int, track['bbox'])
            cls = track['class']
            
            # Если drowning подтвержден - всегда красный
            if self.drowning_confirmed.get(tid, False):
                color = self.colors[2]  # Красный
            else:
                color = self.colors.get(cls, (255, 255, 255))
            
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, self.box_thickness)
            
            # Получаем информацию для отображения
            head_visible = self.behavior_analyzer.is_head_visible(track.get('last_keypoints'))
            arms_up, arms_conf, arm_orient = self.pose_analyzer.analyze_arm_position(track.get('last_keypoints')) if track.get('last_keypoints') is not None else (False, 0.0, 'unknown')
            
            # Иконки состояния
            head_icon = "👤" if head_visible else "🌊"
            arms_icon = "🙌" if arms_up else "  "
            
            # Добавляем индикатор падения
            if len(track['history']) > self.fps // 2:
                is_falling, fall_speed, is_passive = self.behavior_analyzer.analyze_vertical_movement(track['history'])
                fall_indicator = "⬇️" if is_falling and is_passive else "⤵️" if is_falling else ""
            else:
                fall_indicator = ""
            
            # Определяем статус
            class_names = ['SWIM', 'STRUG', 'DROWN']
            
            # Проверяем время в состоянии drowning
            drowning_time = 0
            if tid in self.drowning_start_time:
                drowning_time = current_time - self.drowning_start_time[tid]
            
            # Формируем метку с учетом подтверждения
            if self.drowning_confirmed.get(tid, False):
                # Подтвержденное drowning
                label = f"{head_icon}{arms_icon}{fall_indicator} ID:{tid} DROWN✅ ({drowning_time:.1f}s)"
                # Толстая красная рамка для подтвержденного
                cv2.rectangle(frame, (x1-3, y1-3), (x2+3, y2+3), (0, 0, 255), 4)
            elif cls == 2 and tid in self.drowning_start_time:
                # В процессе подтверждения
                label = f"{head_icon}{arms_icon}{fall_indicator} ID:{tid} DROWN⏳ ({drowning_time:.1f}s)"
            else:
                # Обычный класс
                label = f"{head_icon}{arms_icon}{fall_indicator} ID:{tid} {class_names[cls]}"
            
            # Рисуем фон для текста
            (text_width, text_height), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
            cv2.rectangle(frame, (x1, y1 - text_height - 10), (x1 + text_width + 10, y1), (0, 0, 0), -1)
            cv2.putText(frame, label, (x1 + 5, y1 - 5), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            
            # Если drowning подтвержден, показываем крупное предупреждение
            if self.drowning_confirmed.get(tid, False):
                cv2.putText(frame, f"🚨 CONFIRMED DROWNING {drowning_time:.1f}s", (x1, y2 + 30), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 3)
        
        return frame
    
    def check_alerts(self, current_time):
        """
        ПРОВЕРКА АЛЕРТОВ - только для подтвержденного drowning
        """
        alerts = []
        
        for tid, track in self.tracks.items():
            if track['lost'] > 0:
                continue
            
            # Проверяем, подтверждено ли drowning
            if self.drowning_confirmed.get(tid, False):
                drowning_duration = current_time - self.drowning_start_time.get(tid, current_time)
                
                # Проверяем, не отправляли ли алерт недавно
                last_alert = track.get('last_alert_time', 0)
                if current_time - last_alert > 10:  # Не чаще раза в 10 секунд
                    alerts.append({
                        'track_id': tid,
                        'message': f'🚨 CONFIRMED DROWNING! ({drowning_duration:.1f}s)',
                        'color': (0, 0, 255),
                        'duration': drowning_duration
                    })
                    track['last_alert_time'] = current_time
                    print(f"⚠️ АЛЕРТ: Человек {tid} тонет {drowning_duration:.1f} секунд (ПОДТВЕРЖДЕНО)")
        
        return alerts
# =============================================================================
# УПРОЩЁННЫЙ КЛАСС SmartUnderwaterTester (БЕЗ SLICING)
# =============================================================================

class SmartUnderwaterTester:
    def __init__(self, project_state):
        self.project = project_state
        self.pose_model = None
        self.detection_model = None
        
    def load_models(self):
        if self.pose_model is None:
            print("📥 Загрузка YOLOv8-Pose...")
            self.pose_model = YOLO('yolov8n-pose.pt')
        
        model_path = self.project.state.get('best_model_path')
        if model_path and Path(model_path).exists():
            self.detection_model = YOLO(model_path)
            print(f"✅ Загружена обученная модель: {Path(model_path).name}")
    
    def detect(self, frame, conf_threshold=0.25):
        results = self.pose_model(
            frame, 
            conf=conf_threshold,
            iou=0.5,
            max_det=100,
            verbose=False
        )[0]
        
        detections = []
        keypoints_list = []
        
        if results.keypoints is not None and results.boxes is not None:
            for i, kp in enumerate(results.keypoints.data):
                if i < len(results.boxes):
                    box = results.boxes[i]
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    conf = box.conf[0].item()
                    
                    detections.append([x1, y1, x2, y2, conf, 0])
                    
                    kp_np = kp.cpu().numpy() if hasattr(kp, 'cpu') else np.array(kp)
                    kp_flat = []
                    for j in range(17):
                        if j < len(kp_np):
                            kp_flat.extend([float(kp_np[j][0]), float(kp_np[j][1]), float(kp_np[j][2])])
                        else:
                            kp_flat.extend([0, 0, 0])
                    keypoints_list.append(kp_flat)
        
        return detections, keypoints_list
    
    def test_video(self, video_path, conf_threshold=0.25):
        self.load_models()
        
        cap = cv2.VideoCapture(str(video_path))
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        print(f"\n📹 Видео: {video_path.name}")
        print(f"   {width}x{height}, {fps} fps, {total_frames} кадров")
        
        tracker = SmartUnderwaterTracker(fps=fps)
        
        out_path = self.project.OUTPUTS_DIR / f"underwater_{datetime.now().strftime('%Y%m%d_%H%M%S')}.mp4"
        out = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
        
        frame_count = 0
        alert_count = 0
        start_time = time.time()
        total_detections = 0
        
        print("\n🚀 Обработка подводного видео...")
        
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            frame_count += 1
            
            detections, keypoints_list = self.detect(frame, conf_threshold)
            total_detections += len(detections)
            
            if detections and keypoints_list:
                tracker.update(frame, detections, keypoints_list, time.time())
            
            frame = tracker.draw(frame)
            alerts = tracker.check_alerts(time.time())
            alert_count += len(alerts)
            
            cv2.putText(frame, f"Frame: {frame_count}/{total_frames}", (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
            cv2.putText(frame, f"Tracks: {len(tracker.tracks)}", (10, 60),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
            cv2.putText(frame, f"Dets: {len(detections)}", (10, 90),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
            
            for i, alert in enumerate(alerts[:2]):
                cv2.putText(frame, alert['message'], (10, 120 + i*30),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
            
            out.write(frame)
            
            if frame_count % 30 == 0:
                elapsed = time.time() - start_time
                progress = (frame_count / total_frames) * 100
                
                clear_output(wait=True)
                
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                ax.clear()
                ax.imshow(frame_rgb)
                ax.set_title(f"Подводная съемка | Прогресс: {progress:.1f}% | Тревог: {alert_count}")
                ax.axis('off')
                plt.show()
                
                print(f"   Прогресс: {progress:.1f}% | Треков: {len(tracker.tracks)} | Детекций: {total_detections}")
        
        cap.release()
        out.release()
        
        clear_output(wait=True)
        
        print(f"\n✅ Обработка завершена!")
        print(f"📁 Результат: {out_path}")
        print(f"⚠️ Всего тревог: {alert_count}")
        print(f"📊 Среднее детекций на кадр: {total_detections/frame_count:.1f}")
        print(f"⏱️ Время: {time.time() - start_time:.1f}с")
        
        return out_path

# =============================================================================
# КЛАСС MainMenu
# =============================================================================

class MainMenu:
    def __init__(self):
        self.project = ProjectState()
        self.dm = DatasetManager(self.project)
        self.trainer = ModelTrainer(self.project)
        self.tester = SmartUnderwaterTester(self.project)
        self.output = widgets.Output()
        self.selected_video = None
        self._create_interface()
    
    def _get_status_html(self):
        state = self.project.state
        ds = '✅' if state['dataset_checked'] else '❌'
        cfg = '✅' if state['config_created'] else '❌'
        model = '✅' if state['model_trained'] else '❌'
        
        model_info = ""
        if state['best_model_path'] and Path(state['best_model_path']).exists():
            model_path = Path(state['best_model_path'])
            size = model_path.stat().st_size / (1024*1024)
            model_info = f"<br><small>📁 {model_path.name} ({size:.1f} MB)</small>"
        
        return f"""
        <div style='border:2px solid #4CAF50; padding:15px; border-radius:10px; margin:10px 0; 
                    background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);'>
            <h3 style='margin:0; color:#333; display: flex; align-items: center;'>
                <span style='font-size:24px; margin-right:10px;'>📊</span> 
                СТАТУС ПРОЕКТА
            </h3>
            <table style='width:100%; margin-top:10px; border-collapse: collapse;'>
                <tr style='border-bottom:1px solid #ddd;'>
                    <td style='padding:8px;'><b>📁 Датасет проверен:</b></td>
                    <td style='padding:8px; font-size:20px;'>{ds}</td>
                </tr>
                <tr style='border-bottom:1px solid #ddd;'>
                    <td style='padding:8px;'><b>📝 Конфиг создан:</b></td>
                    <td style='padding:8px; font-size:20px;'>{cfg}</td>
                </tr>
                <tr>
                    <td style='padding:8px;'><b>🤖 Модель обучена:</b></td>
                    <td style='padding:8px; font-size:20px;'>{model}</td>
                </tr>
            </table>
            {model_info}
            <div style='margin-top:10px; font-size:12px; color:#666;'>
                🕐 Последнее обновление: {time.strftime("%H:%M:%S")}
            </div>
        </div>
        """
    
    def _create_interface(self):
        self.title = widgets.HTML("""
        <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                    padding: 25px; border-radius: 15px; color: white; text-align: center;
                    box-shadow: 0 4px 6px rgba(0,0,0,0.1);'>
            <h1 style='margin:0; font-size:2.2em;'>🌊 ПОДВОДНАЯ СИСТЕМА ОБНАРУЖЕНИЯ УТОПАЮЩИХ</h1>
            <p style='margin:10px 0 0 0; opacity:0.9; font-size:1.1em;'>
                YOLOv8-Pose | Анализ 17 ключевых точек | Умное поведение под водой
            </p>
        </div>
        """)
        
        self.status = widgets.HTML(self._get_status_html())
        
        self.refresh = widgets.Button(
            description='🔄 Обновить статус',
            button_style='info',
            layout=widgets.Layout(width='150px', height='36px')
        )
        self.refresh.on_click(self._refresh)
        
        self.tab = widgets.Tab()
        self._create_tabs()
        
        self.menu = widgets.VBox([
            self.title,
            widgets.HBox([self.refresh], layout=widgets.Layout(justify_content='flex-end')),
            self.status,
            self.tab,
            self.output
        ])
    
    def _refresh(self, b):
        with self.output:
            clear_output()
            self.status.value = self._get_status_html()
            print("✅ Статус обновлен")
    
    def _create_tabs(self):
        self.tab.children = [
            self._create_dataset_tab(),
            self._create_training_tab(),
            self._create_testing_tab(),
            self._create_info_tab()
        ]
        self.tab.set_title(0, '📁 Датасет')
        self.tab.set_title(1, '🚀 Обучение')
        self.tab.set_title(2, '🌊 Тест')
        self.tab.set_title(3, 'ℹ️ Инфо')
    
    def _create_dataset_tab(self):
        header = widgets.HTML("""
        <div style='background-color:#e3f2fd; padding:15px; border-radius:10px; margin-bottom:15px;'>
            <h3 style='margin:0; color:#1565c0;'>📁 УПРАВЛЕНИЕ ДАТАСЕТОМ</h3>
            <p style='margin:5px 0 0 0; color:#0d47a1;'>
                Проверьте структуру датасета и создайте конфигурацию для обучения
            </p>
        </div>
        """)
        
        structure_info = widgets.HTML("""
        <div style='background:#f5f5f5; padding:15px; border-radius:8px; margin:10px 0; 
                    border-left:4px solid #2196f3;'>
            <b style='color:#1565c0;'>📁 Ожидаемая структура:</b>
            <pre style='background:#fff; padding:10px; border-radius:5px; margin:10px 0;'>
dataset/
├── images/
│   ├── train/    (изображения для обучения)
│   └── val/      (изображения для валидации)
└── labels/
    ├── train/    (аннотации для обучения)
    └── val/      (аннотации для валидации)
            </pre>
            <p style='color:#666; margin:0;'>
                <i>Классы: swimming (0), struggling (1), drowning (2)</i>
            </p>
        </div>
        """)
        
        btn_layout = widgets.Layout(width='180px', height='50px')
        
        check_btn = widgets.Button(
            description='🔍 Проверить структуру',
            button_style='info',
            layout=btn_layout
        )
        
        config_btn = widgets.Button(
            description='📝 Создать data.yaml',
            button_style='success',
            layout=btn_layout
        )
        
        show_btn = widgets.Button(
            description='👁 Показать примеры',
            button_style='warning',
            layout=btn_layout
        )
        
        split = widgets.Dropdown(
            options=['train', 'val'],
            value='train',
            description='Раздел:',
            layout=widgets.Layout(width='200px')
        )
        
        num = widgets.IntSlider(
            value=5,
            min=1,
            max=10,
            step=1,
            description='Количество:',
            layout=widgets.Layout(width='300px')
        )
        
        out = widgets.Output()
        
        def on_check(b):
            with out:
                clear_output()
                self.dm.check_dataset_structure()
                self.status.value = self._get_status_html()
        
        def on_config(b):
            with out:
                clear_output()
                self.dm.create_data_yaml()
                self.status.value = self._get_status_html()
        
        def on_show(b):
            with out:
                clear_output()
                self.dm.show_samples(split.value, num.value)
        
        check_btn.on_click(on_check)
        config_btn.on_click(on_config)
        show_btn.on_click(on_show)
        
        return widgets.VBox([
            header,
            structure_info,
            widgets.HBox([check_btn, config_btn, show_btn], 
                        layout=widgets.Layout(justify_content='space-around', margin='10px 0')),
            widgets.HBox([split, num], layout=widgets.Layout(margin='10px 0')),
            out
        ])
    
    def _create_training_tab(self):
        header = widgets.HTML("""
        <div style='background-color:#fff3e0; padding:15px; border-radius:10px; margin-bottom:15px;'>
            <h3 style='margin:0; color:#e65100;'>🚀 ОБУЧЕНИЕ МОДЕЛИ</h3>
            <p style='margin:5px 0 0 0; color:#bf360c;'>
                Настройте параметры и начните обучение. Процесс может занять несколько часов.
            </p>
        </div>
        """)
        
        size_box = widgets.VBox([
            widgets.HTML("<b style='font-size:14px;'>1. Выберите размер модели:</b>"),
            widgets.RadioButtons(
                options=[
                    ('⚡ nano (n) - САМАЯ БЫСТРАЯ (1-2 часа)', 'n'),
                    ('⚖️ small (s) - ОПТИМАЛЬНЫЙ БАЛАНС (4-6 часов)', 's'),
                    ('🎯 medium (m) - САМАЯ ТОЧНАЯ (8-12 часов)', 'm')
                ],
                value='s',
                layout=widgets.Layout(margin='5px 0 10px 20px')
            )
        ])
        
        epochs_box = widgets.VBox([
            widgets.HTML("<b style='font-size:14px;'>2. Количество эпох:</b>"),
            widgets.HTML("""
            <div style='margin-left:20px; color:#666; font-size:12px;'>
                • 50 эпох - быстрый тест<br>
                • 100 эпох - оптимально ⭐<br>
                • 150-200 эпох - максимальное качество
            </div>
            """),
            widgets.IntSlider(
                value=100,
                min=10,
                max=300,
                step=10,
                description='Эпохи:',
                style={'description_width': 'initial'},
                layout=widgets.Layout(width='500px', margin='5px 0 10px 20px')
            )
        ])
        
        batch_box = widgets.VBox([
            widgets.HTML("<b style='font-size:14px;'>3. Размер батча (batch size):</b>"),
            widgets.HTML("""
            <div style='margin-left:20px; color:#666; font-size:12px;'>
                • 8-16 - для 4-6GB GPU<br>
                • 16-32 - для 8GB+ GPU ⭐
            </div>
            """),
            widgets.IntSlider(
                value=16,
                min=4,
                max=64,
                step=4,
                description='Batch:',
                style={'description_width': 'initial'},
                layout=widgets.Layout(width='500px', margin='5px 0 10px 20px')
            )
        ])
        
        imgsz_box = widgets.VBox([
            widgets.HTML("<b style='font-size:14px;'>4. Размер входных изображений:</b>"),
            widgets.HTML("""
            <div style='margin-left:20px; color:#666; font-size:12px;'>
                • 416 - быстрее<br>
                • 640 - стандарт ⭐
            </div>
            """),
            widgets.Dropdown(
                options=[416, 512, 640],
                value=640,
                description='Размер:',
                style={'description_width': 'initial'},
                layout=widgets.Layout(width='200px', margin='5px 0 10px 20px')
            )
        ])
        
        train_btn = widgets.Button(
            description='🚀 НАЧАТЬ ОБУЧЕНИЕ',
            button_style='danger',
            layout=widgets.Layout(width='200px', height='50px', font_weight='bold')
        )
        
        evaluate_btn = widgets.Button(
            description='📊 ОЦЕНИТЬ МОДЕЛЬ',
            button_style='info',
            layout=widgets.Layout(width='200px', height='50px')
        )
        
        out = widgets.Output()
        
        def on_train(b):
            with out:
                clear_output()
                self.trainer.train_model(
                    model_size=size_box.children[1].value,
                    epochs=epochs_box.children[2].value,
                    batch_size=batch_box.children[2].value,
                    imgsz=imgsz_box.children[2].value
                )
        
        def on_evaluate(b):
            with out:
                clear_output()
                self.trainer.evaluate_model()
        
        train_btn.on_click(on_train)
        evaluate_btn.on_click(on_evaluate)
        
        return widgets.VBox([
            header,
            size_box,
            epochs_box,
            batch_box,
            imgsz_box,
            widgets.HBox([train_btn, evaluate_btn], 
                        layout=widgets.Layout(justify_content='center', margin='20px 0')),
            out
        ])
    
    def _create_testing_tab(self):
        """Упрощённая вкладка тестирования для подводной съёмки"""
        
        header = widgets.HTML("""
        <div style='background-color:#e3f2fd; padding:15px; border-radius:10px; margin-bottom:15px;'>
            <h3 style='margin:0; color:#0d47a1;'>🌊 ТЕСТИРОВАНИЕ ПОДВОДНОЙ МОДЕЛИ</h3>
            <p style='margin:5px 0 0 0; color:#1565c0;'>
                Анализ поведения под водой: видимость головы, движение, время без движения
            </p>
        </div>
        """)
        
        # Папка для видео
        video_folder = Path("videos")
        video_folder.mkdir(exist_ok=True)
        
        def get_videos():
            videos = []
            for ext in ['*.mp4', '*.avi', '*.mov', '*.mkv']:
                videos.extend(video_folder.glob(ext))
            return sorted([v.name for v in videos])
        
        # Способ 1: указать путь
        path_box = widgets.VBox([
            widgets.HTML("<b>📁 Способ 1: Укажите путь к видео</b>"),
            widgets.HBox([
                widgets.Text(
                    value='',
                    placeholder='Например: C:\\video.mp4 или videos\\test.mp4',
                    layout=widgets.Layout(width='500px')
                ),
                widgets.Button(
                    description='🔍 Проверить',
                    button_style='info',
                    layout=widgets.Layout(width='100px')
                )
            ])
        ])
        
        # Способ 2: выбрать из папки
        list_box = widgets.VBox([
            widgets.HTML("<b>📋 Способ 2: Выберите из папки videos</b>"),
            widgets.HBox([
                widgets.Select(
                    options=get_videos() if get_videos() else ['(нет видео)'],
                    layout=widgets.Layout(width='400px', height='120px')
                ),
                widgets.Button(
                    description='🔄 Обновить',
                    layout=widgets.Layout(width='100px')
                )
            ])
        ])
        
        # Информация о выбранном видео
        file_info = widgets.HTML(
            value="<i style='color:gray;'>Видео не выбрано</i>",
            layout=widgets.Layout(margin='10px 0')
        )
        
        # Параметры
        params_box = widgets.VBox([
            widgets.HTML("<b>⚙️ Параметры тестирования:</b>"),
            widgets.FloatSlider(
                value=0.25,
                min=0.1,
                max=0.5,
                step=0.05,
                description='Порог:',
                style={'description_width': 'initial'},
                layout=widgets.Layout(width='400px')
            )
        ])
        
        # Кнопки
        select_btn = widgets.Button(
            description='📌 ВЫБРАТЬ ВИДЕО',
            button_style='success',
            disabled=True,
            layout=widgets.Layout(width='200px', height='40px')
        )
        
        test_btn = widgets.Button(
            description='▶️ ЗАПУСТИТЬ ТЕСТ',
            button_style='primary',
            disabled=True,
            layout=widgets.Layout(width='200px', height='40px')
        )
        
        out = widgets.Output()
        
        path_input = path_box.children[1].children[0]
        check_btn = path_box.children[1].children[1]
        video_list = list_box.children[1].children[0]
        refresh_btn = list_box.children[1].children[1]
        conf_slider = params_box.children[1]
        
        def on_check(b):
            with out:
                clear_output()
                p = Path(path_input.value.strip().strip('"').strip("'"))
                if p.exists():
                    size = p.stat().st_size / (1024*1024)
                    print(f"✅ Видео найдено: {p.name}")
                    print(f"📊 Размер: {size:.1f} MB")
                    file_info.value = f"<b style='color:green;'>✅ Выбрано: {p.name}</b>"
                    self.selected_video = p
                    select_btn.disabled = False
                else:
                    print(f"❌ Файл не найден: {p}")
                    file_info.value = "<b style='color:red;'>❌ Файл не найден</b>"
        
        def on_select(b):
            with out:
                clear_output()
                if self.selected_video and self.selected_video.exists():
                    size = self.selected_video.stat().st_size / (1024*1024)
                    print("="*60)
                    print("🎬 ВИДЕО ВЫБРАНО")
                    print("="*60)
                    print(f"📁 Файл: {self.selected_video.name}")
                    print(f"📊 Размер: {size:.1f} MB")
                    print("="*60)
                    file_info.value = f"<b style='color:green;'>✅ Готово: {self.selected_video.name}</b>"
                    test_btn.disabled = False
                else:
                    print("❌ Файл не найден!")
        
        def on_test(b):
            with out:
                clear_output()
                if self.selected_video:
                    self.tester.test_video(self.selected_video, conf_slider.value)
                else:
                    print("❌ Сначала выберите видео!")
        
        def on_refresh(b):
            with out:
                clear_output()
                videos = get_videos()
                video_list.options = videos if videos else ['(нет видео)']
                print(f"🔄 Найдено видео: {len(videos)}")
        
        def on_list_select(change):
            if change['new'] and change['new'] != '(нет видео)':
                path_input.value = str(video_folder / change['new'])
                file_info.value = f"<b>Выбрано:</b> {change['new']}"
        
        check_btn.on_click(on_check)
        select_btn.on_click(on_select)
        test_btn.on_click(on_test)
        refresh_btn.on_click(on_refresh)
        video_list.observe(on_list_select, names='value')
        
        return widgets.VBox([
            header,
            path_box,
            list_box,
            file_info,
            params_box,
            widgets.HBox([select_btn, test_btn], 
                        layout=widgets.Layout(justify_content='center', margin='20px 0')),
            out
        ])
    
    def _create_info_tab(self):
        html = """
        <div style='padding:20px; background:#f9f9f9; border-radius:10px;'>
            <h3 style='color:#333; border-bottom:2px solid #4CAF50; padding-bottom:10px;'>
                ℹ️ О ПОДВОДНОЙ СИСТЕМЕ
            </h3>
            
            <div style='margin:20px 0;'>
                <h4 style='color:#1565c0;'>📊 КЛАССЫ:</h4>
                <table style='width:100%; border-collapse: collapse;'>
                    <tr style='background:#e8f5e8;'>
                        <td style='padding:10px;'><span style='color:green; font-size:20px;'>■</span></td>
                        <td style='padding:10px;'><b>swimming (0)</b></td>
                        <td style='padding:10px;'>Нормальное плавание</td>
                    </tr>
                    <tr style='background:#fff3e0;'>
                        <td style='padding:10px;'><span style='color:orange; font-size:20px;'>■</span></td>
                        <td style='padding:10px;'><b>struggling (1)</b></td>
                        <td style='padding:10px;'>Борьба, необычное поведение</td>
                    </tr>
                    <tr style='background:#ffebee;'>
                        <td style='padding:10px;'><span style='color:red; font-size:20px;'>■</span></td>
                        <td style='padding:10px;'><b>drowning (2)</b></td>
                        <td style='padding:10px;'>Утопление (тревога)</td>
                    </tr>
                </table>
            </div>
            
            <div style='margin:20px 0;'>
                <h4 style='color:#1565c0;'>🧠 ЛОГИКА ПОД ВОДОЙ:</h4>
                <ul style='list-style-type:none; padding:0;'>
                    <li style='margin:10px 0; padding:10px; background:#e3f2fd; border-radius:5px;'>
                        <b>👤 Голова видна → на поверхности</b><br>
                        - Движется: swimming<br>
                        - Не движется 5с: drowning
                    </li>
                    <li style='margin:10px 0; padding:10px; background:#e0f2f1; border-radius:5px;'>
                        <b>🌊 Под водой (голова не видна)</b><br>
                        - Стоит на дне: swimming<br>
                        - Плывет: struggling<br>
                        - Нет движения 5с: drowning
                    </li>
                </ul>
            </div>
            
            <div style='margin:20px 0;'>
                <h4 style='color:#1565c0;'>📁 СТРУКТУРА ДАТАСЕТА:</h4>
                <pre style='background:#fff; padding:15px; border-radius:5px; border:1px solid #ddd;'>
dataset/
├── images/
│   ├── train/    # Изображения для обучения
│   └── val/      # Изображения для валидации
└── labels/
    ├── train/    # Аннотации для обучения
    └── val/      # Аннотации для валидации
                </pre>
            </div>
            
            <div style='margin:20px 0;'>
                <h4 style='color:#1565c0;'>⚙️ РЕКОМЕНДАЦИИ:</h4>
                <ul>
                    <li><b>Для быстрого теста:</b> nano, 50 эпох</li>
                    <li><b>Оптимально:</b> small, 100 эпох ⭐</li>
                    <li><b>Для максимальной точности:</b> medium, 150 эпох</li>
                </ul>
            </div>
        </div>
        """
        return widgets.HTML(html)
    
    def show(self):
        display(self.menu)

# =============================================================================
# ЗАПУСК МЕНЮ
# =============================================================================

print("\n" + "="*70)
print("🌊 ЗАПУСК ПОДВОДНОЙ СИСТЕМЫ ОБНАРУЖЕНИЯ УТОПАЮЩИХ")
print("="*70)
print("\n📊 КОМПОНЕНТЫ:")
print("   • YOLOv8-Pose - анализ 17 ключевых точек")
print("   • UnderwaterBehaviorAnalyzer - умная логика поведения")
print("   • SmartUnderwaterTracker - интеллектуальный трекинг")
print("\n" + "="*70)

# Создаем и показываем меню
menu = MainMenu()
menu.show()

print("\n✅ СИСТЕМА ГОТОВА К РАБОТЕ")
print("="*70)
print("\n📋 ИСПОЛЬЗУЙТЕ ВКЛАДКИ:")
print("   1. 📁 Датасет - проверьте структуру и создайте конфиг")
print("   2. 🚀 Обучение - настройте параметры и обучите модель")
print("   3. 🌊 Тест - протестируйте на подводном видео")
print("   4. ℹ️ Инфо - справка по системе")

📚 Библиотеки импортированы
🔥 PyTorch version: 2.7.1+cu118
🎮 CUDA available: True
🎮 GPU: NVIDIA GeForce RTX 3050

🌊 ЗАПУСК ПОДВОДНОЙ СИСТЕМЫ ОБНАРУЖЕНИЯ УТОПАЮЩИХ

📊 КОМПОНЕНТЫ:
   • YOLOv8-Pose - анализ 17 ключевых точек
   • UnderwaterBehaviorAnalyzer - умная логика поведения
   • SmartUnderwaterTracker - интеллектуальный трекинг




✅ СИСТЕМА ГОТОВА К РАБОТЕ

📋 ИСПОЛЬЗУЙТЕ ВКЛАДКИ:
   1. 📁 Датасет - проверьте структуру и создайте конфиг
   2. 🚀 Обучение - настройте параметры и обучите модель
   3. 🌊 Тест - протестируйте на подводном видео
   4. ℹ️ Инфо - справка по системе
